Bibliotecas necessárias.

In [ ]:
# Importar bibliotecas necessárias
import pandas as pd
import re
import gc
from google.colab import userdata

Funções auxiliares.

In [ ]:
# Função para remover a parte numérica inicial e os possíveis sublinhado que a segue.
def clean_category(category):
  # Dicionário de correções de termos jurídicos.
  corrections_dict = {
      'tributario': 'tributário'
  }
  cleaned_numeric_part = re.sub(r'^\d+_?', '', category)

  # Substituir todos os sublinhados por espaços
  final_string = cleaned_numeric_part.replace('_', ' ')

  # Aplicar correções.
  words = final_string.split()
  corrected_words = []
  for word in words:
    # Verifica se a palavra está no dicionário de correções e aplica
    corrected_words.append(corrections_dict.get(word, word))
  final_string = ' '.join(corrected_words)

  # Converter para modo título: primeira letra de cada palavra maiúscula, as demais minúsculas,
  # com as exceções de alguns artigos e preosições.
  lowercase_exceptions = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
  processed_words = []
  for word in final_string.split():
    if word.lower() in lowercase_exceptions:
      processed_words.append(word.lower())
    else:
      processed_words.append(word.title())
  # Retorno.
  return ' '.join(processed_words)

# Função que recebe a url e retorna um Dataframe.
def load_dataframe(url):
   return pd.read_json(url, lines=True)

# Função recuperar todas as categorias de um Dataframe de forma distinta.
def get_unique_categories(df):
  return df['category'].unique()

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [ ]:
# Dataframe questions.
def load_questions():
  # URL das questões.
  url_questions = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench/question.jsonl"

  # Carregar um Dataframe com as perguntas a partir do arquivo .jsonl.
  df_questions = load_dataframe(url_questions)

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df_questions.insert(0, 'question', range(1, len(df_questions)+1))

  # Realizar limpeza no campo das categorias.
  df_questions['category'] = df_questions['category'].apply(clean_category)

  # retorno da Dataframe.
  return df_questions

# Dataframe guidelines.
def load_guidelines():
  # URL das respostas de referência.
  url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"
  # Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
  return load_dataframe(url_guidelines)

# Dataframe da junção dos DataFrames df_questions e df_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia juntas.
# E, também personalizada as colunas que serão apresentadas.
def load_questions_and_guidelines():
  # Carregar os Dataframes.
  df_questions = load_questions()
  df_guidelines = load_guidelines()

  # Chave do inner join
  key = 'question_id'
  # Colunas selecionar para projeção no Dataframe.
  columns=['question', 'question_id', 'category', 'statement', 'turns', 'system', 'choices']

  # Junção e retorno do Dataframe
  return merge_dataframes(df_questions, df_guidelines, key, columns)

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_questions_and_guidelines(question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df_my_questions = load_questions_and_guidelines().iloc[question_min:question_max].copy()

  # Realizar limpeza das informações da coluna turns, pois é um array.
  # E, só preciso da infomação, quando houver, da primeira posição.
  df_my_questions['turns'] = df_my_questions['turns'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else '')

  # Limpeza também a coluna choices, pois é um dicionário que só preciso do campo turns.
  df_my_questions['choices'] = df_my_questions['choices'].apply(lambda x: x[0]['turns'][0] if isinstance(x, list) and len(x) > 0 and 'turns' in x[0] and isinstance(x[0]['turns'], list) and len(x[0]['turns']) > 0 else '')

  # Retornar dataframe ajustado.
  return df_my_questions

# Excluir Dataframes desnecessários.
def delete_obsoletes_objects():
  # Excluir Dataframes que não são mais necessários, apenas se existirem no escopo global.
  if 'df_questions' in globals():
    del df_questions
  if 'df_guidelines' in globals():
    del df_guidelines
  gc.collect()

Respostas de referências às questões anteriores e disponíveis no mesmo repositório do github.

In [ ]:
# Meu sub-grupo de questões e respostas.
# Subconjunto das questões de 31 a 40.
question_min = 31
question_max = 40
df_my_questions = load_my_questions_and_guidelines(question_min, question_max)

Função para definir o nível de complexidade das questões.

In [ ]:
# Classificar os níveis das perguntas.
# Após analise manual e também com ajuda de algumas IAs percebi que todas as questões com texto
# no turns do subconjunto que peguei ficou no nível 3.
# E, atribui o nível 1 a quem não tem texto algum em turns.
# Os níveis são:
#   1: Estagiário;
#   2: Analista;
#   3: Juiz;
#   4: Ministro.
def classify_difficulty_level():
  for index, row in df_my_questions.iterrows():
      # Verifica se a coluna 'turns' está vazia ou contém apenas espaços em branco
      if not row['turns'] or str(row['turns']).strip() == '':
          df_my_questions.loc[index, 'level'] = 1
      else:
          df_my_questions.loc[index, 'level'] = 3
  # Converter a coluna 'level' para o tipo inteiro após o preenchimento
  df_my_questions['level'] = df_my_questions['level'].astype(int)

Chamar a função que adiciona o nível de dificuldade no dataframe df_my_questions.

In [ ]:
classify_difficulty_level()

Liberar espeço não necessário em memória.

In [ ]:
delete_obsoletes_objects()